# 🗄️ Day 5B — Build the Tax Knowledge Vector Store
**Agentic AI for Tax Technologists · 12-Day Program**

---

## 📖 Where We Are

Notebook 5A showed you what embeddings are and proved that semantically similar tax phrases cluster together.
Now we store those embeddings in a **vector store** — an index that finds the most similar documents in milliseconds.

**What you will build:**
- A `NumpyVectorStore` — a pure-Python vector store using cosine similarity
- 20 pre-chunked tax knowledge entries (GST + TDS + Income Tax)
- Each chunk embedded and stored with rich metadata
- 5 semantic queries run against the store
- Side-by-side comparison: semantic retrieval vs keyword matching

> **Why NumpyVectorStore and not ChromaDB?**  
> ChromaDB has compiled C extensions that do not install in Pyodide or Python 3.13.  
> For 20 chunks, numpy cosine similarity is functionally identical — and actually faster  
> than ChromaDB's HNSW index, which only pays off above several thousand vectors.  
> The API is designed to match ChromaDB's `.add()` / `.query()` interface so Day 6  
> notebooks can swap in the real ChromaDB with a one-line change if needed.

---
## ⏱️ Time: 60 minutes

---
## ⚙️ Step 1: Install & Import

Only `openai` and `numpy` — both work in Pyodide, Python 3.13, 3.12, and 3.11.

In [ ]:
%pip install openai numpy --quiet

In [ ]:
import os
import re
import json
import numpy as np
import datetime
from getpass import getpass
from openai import AzureOpenAI

print("✅ Imports OK")

---
## 🔑 Step 2: Configure Azure OpenAI

> **Two deployments needed:**
> - Chat deployment (e.g. `gpt-4o`) — for generation in Notebook 5C
> - Embeddings deployment (e.g. `text-embedding-ada-002`) — for this notebook

In [ ]:
AZURE_OPENAI_ENDPOINT      = input("Endpoint (e.g. https://xxx.openai.azure.com/): ").strip()
AZURE_OPENAI_API_KEY       = getpass("API Key: ")
AZURE_DEPLOYMENT_NAME      = input("Chat deployment name (e.g. gpt-4o): ").strip()
AZURE_EMBEDDING_DEPLOYMENT = input("Embeddings deployment name (e.g. text-embedding-ada-002): ").strip()
AZURE_API_VERSION          = "2024-08-01-preview"

client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
    api_version    = AZURE_API_VERSION,
)
print("✅ Azure OpenAI client initialised successfully!")

---
## 🔧 Step 3: NumpyVectorStore

A pure-Python vector store. Stores embeddings as a numpy matrix and retrieves
by cosine similarity. The `.add()` and `.query()` API mirrors ChromaDB's
interface intentionally — swap in ChromaDB with one line if you need persistence
or scale beyond a few thousand chunks.

In [ ]:
class NumpyVectorStore:
    """
    Lightweight in-memory vector store using numpy cosine similarity.

    API mirrors ChromaDB's .add() and .query() so the rest of
    the notebook — and Notebook 5C — requires zero changes.

    Performance: O(n) scan over all vectors. Fast enough for up to
    ~50,000 chunks. Beyond that, swap in ChromaDB with HNSW indexing.
    """

    def __init__(self, name: str = "collection"):
        self.name       = name
        self.ids        = []
        self.documents  = []
        self.embeddings = []   # list of lists; converted to matrix at query time
        self.metadatas  = []

    # ── Write ────────────────────────────────────────────────────────────────
    def add(self, ids, documents, embeddings, metadatas):
        """
        Add one or more chunks.
        Accepts both single items and lists — same as ChromaDB.
        """
        if isinstance(ids, str):
            ids, documents, embeddings, metadatas = [ids], [documents], [embeddings], [metadatas]
        for _id in ids:
            if _id in self.ids:
                raise ValueError(f"ID '{_id}' already exists. Use .update() to replace.")
        self.ids.extend(ids)
        self.documents.extend(documents)
        self.embeddings.extend(embeddings)
        self.metadatas.extend(metadatas)

    def count(self) -> int:
        return len(self.ids)

    # ── Read ─────────────────────────────────────────────────────────────────
    def query(self, query_embeddings, n_results: int = 3, where: dict = None):
        """
        Find the n_results most similar documents to query_embeddings[0].
        Optionally filter by metadata fields via `where` dict.
        Returns a dict matching ChromaDB's response format.
        """
        if not self.embeddings:
            raise ValueError("Vector store is empty. Call .add() first.")

        q = np.array(query_embeddings[0], dtype=np.float32)
        q_norm = q / (np.linalg.norm(q) + 1e-10)

        # Apply metadata filter if provided
        indices = list(range(len(self.ids)))
        if where:
            indices = [
                i for i in indices
                if all(self.metadatas[i].get(k) == v for k, v in where.items())
            ]

        if not indices:
            return {"ids": [[]], "documents": [[]], "metadatas": [[]], "distances": [[]]}

        # Build matrix from filtered indices and compute cosine similarities
        matrix = np.array([self.embeddings[i] for i in indices], dtype=np.float32)
        norms  = np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-10
        sims   = (matrix / norms) @ q_norm          # cosine similarity, shape (n_filtered,)

        # Top-k: highest similarity = lowest distance (distance = 1 − similarity)
        k       = min(n_results, len(indices))
        top_pos = np.argsort(sims)[::-1][:k]        # positions in filtered list
        top_idx = [indices[p] for p in top_pos]     # original store indices

        return {
            "ids":       [[self.ids[i]       for i in top_idx]],
            "documents": [[self.documents[i] for i in top_idx]],
            "metadatas": [[self.metadatas[i] for i in top_idx]],
            "distances": [[round(float(1 - sims[p]), 4) for p in top_pos]],
        }


print("✅ NumpyVectorStore defined")
print("   Works in: Pyodide · Python 3.11 · 3.12 · 3.13")
print("   API-compatible with ChromaDB .add() / .query()")

---
## 📚 Step 4: The Tax Knowledge Chunks

20 pre-chunked entries across GST, TDS, and Income Tax — each mimicking what
a section-aware splitter would produce from a real circular or notification,
complete with source metadata.

In [ ]:
TAX_KNOWLEDGE_CHUNKS = [

    # ── GST — Exports ────────────────────────────────────────────────────────
    {
        "id": "gst_export_001",
        "text": (
            "Export of services is treated as zero-rated supply under Section 16(1) of the "
            "IGST Act 2017. A registered supplier may export without payment of IGST if a "
            "Letter of Undertaking (LUT) has been filed under Rule 96A. Alternatively, the "
            "supplier may export on payment of IGST and claim a refund under Section 54 of "
            "the CGST Act. LUT is valid for the entire financial year from the date of filing."
        ),
        "metadata": {"source": "IGST Act 2017", "section": "16(1)", "type": "legislation", "topic": "exports", "year": "2017"}
    },
    {
        "id": "gst_export_002",
        "text": (
            "Place of supply for export of services is determined under Section 13 of the IGST "
            "Act. When the recipient is located outside India, the place of supply is the "
            "location of the recipient. IT/ITES services provided to overseas clients qualify "
            "as export of services, subject to: (a) supplier is in India, (b) recipient is "
            "outside India, (c) payment received in convertible foreign exchange, "
            "(d) supplier and recipient are not merely establishments of the same person."
        ),
        "metadata": {"source": "IGST Act 2017", "section": "13", "type": "legislation", "topic": "exports", "year": "2017"}
    },
    {
        "id": "gst_notification_01_2023",
        "text": (
            "Notification No. 01/2023-Integrated Tax (Rate) clarifies the GST rate schedule for "
            "IT and ITES services. Cloud computing services, SaaS subscriptions, and data "
            "processing services supplied by Indian entities to overseas recipients qualify as "
            "export of services and are zero-rated if LUT is filed. If LUT is not filed, IGST "
            "at 18% applies and refund may be claimed. Domestic B2B supply of these services "
            "attracts 18% GST (9% CGST + 9% SGST for intra-state, 18% IGST for inter-state)."
        ),
        "metadata": {"source": "Notification 01/2023-IT(Rate)", "section": "4", "type": "notification", "topic": "exports", "year": "2023"}
    },

    # ── GST — Composition Scheme ─────────────────────────────────────────────
    {
        "id": "gst_composition_001",
        "text": (
            "Section 10 of the CGST Act 2017 provides for the composition scheme. Registered "
            "suppliers with aggregate turnover not exceeding Rs 1.5 crore (Rs 75 lakh for "
            "special category states) may opt for composition levy. Composition dealers pay "
            "tax at 1% of turnover for traders, 2% for manufacturers, and 5% for restaurants. "
            "Composition dealers cannot collect tax from recipients, cannot issue tax invoices, "
            "and cannot avail input tax credit. They must issue a Bill of Supply instead."
        ),
        "metadata": {"source": "CGST Act 2017", "section": "10", "type": "legislation", "topic": "composition_scheme", "year": "2017"}
    },

    # ── GST — Input Tax Credit ───────────────────────────────────────────────
    {
        "id": "gst_itc_001",
        "text": (
            "Input Tax Credit (ITC) under Section 16 of the CGST Act can be claimed if: "
            "(a) the supplier has filed returns and paid the tax, "
            "(b) the recipient holds a valid tax invoice, "
            "(c) the goods or services have been received, and "
            "(d) the tax has been paid to the government. "
            "Section 17(5) lists blocked credits including motor vehicles for personal use, "
            "food and beverages, beauty treatment, health services, and club memberships. "
            "ITC cannot be claimed after filing of the annual return for the relevant financial year."
        ),
        "metadata": {"source": "CGST Act 2017", "section": "16", "type": "legislation", "topic": "input_tax_credit", "year": "2017"}
    },

    # ── GST — Reverse Charge ─────────────────────────────────────────────────
    {
        "id": "gst_rcm_001",
        "text": (
            "Reverse Charge Mechanism (RCM) under Section 9(3) of the CGST Act requires the "
            "recipient to pay GST instead of the supplier in notified categories. Key RCM "
            "notifications include: legal services by an advocate (Notification 13/2017-CT), "
            "services by a director to a company, goods transport agency (GTA) services, "
            "import of services from overseas suppliers, and security services supplied by an "
            "individual to a body corporate. RCM applies at the standard rate for the service."
        ),
        "metadata": {"source": "CGST Act 2017", "section": "9(3)", "type": "legislation", "topic": "reverse_charge", "year": "2017"}
    },

    # ── GST — Registration ───────────────────────────────────────────────────
    {
        "id": "gst_registration_001",
        "text": (
            "GST registration is mandatory when aggregate turnover exceeds Rs 20 lakh for "
            "service providers (Rs 10 lakh in special category states) and Rs 40 lakh for "
            "goods suppliers. Inter-state suppliers must register regardless of turnover. "
            "E-commerce operators and persons supplying through e-commerce operators must "
            "register regardless of turnover. Voluntary registration is permitted below threshold."
        ),
        "metadata": {"source": "CGST Act 2017", "section": "22", "type": "legislation", "topic": "registration", "year": "2017"}
    },

    # ── GST — Filing ─────────────────────────────────────────────────────────
    {
        "id": "gst_gstr1_001",
        "text": (
            "GSTR-1 is the return for outward supplies filed by registered taxpayers. Monthly "
            "filers with turnover above Rs 5 crore must file by the 11th of the following month. "
            "Quarterly filers under QRMP scheme file by the 13th of the month following each "
            "quarter. GSTR-1 includes B2B invoices, B2C summary, exports, debit/credit notes, "
            "and HSN summary. Late filing attracts Rs 50/day (Rs 20/day for nil returns)."
        ),
        "metadata": {"source": "CGST Rules 2017", "section": "Rule 59", "type": "rule", "topic": "filing", "year": "2017"}
    },
    {
        "id": "gst_gstr3b_001",
        "text": (
            "GSTR-3B is the monthly summary return for payment of GST liability. Monthly "
            "taxpayers file by the 20th of the following month. Under QRMP scheme, GSTR-3B is "
            "filed quarterly — by the 22nd for Category-I states and 24th for Category-II states. "
            "GSTR-3B covers outward supplies, inward supplies liable to RCM, eligible ITC, "
            "and net tax payable. Payment must accompany filing."
        ),
        "metadata": {"source": "CGST Rules 2017", "section": "Rule 61", "type": "rule", "topic": "filing", "year": "2017"}
    },

    # ── TDS ──────────────────────────────────────────────────────────────────
    {
        "id": "tds_194c_001",
        "text": (
            "Section 194C of the Income Tax Act requires TDS deduction on payments to "
            "contractors and sub-contractors. TDS rate is 1% for payments to individuals "
            "and HUFs, and 2% for companies and firms. TDS is deducted when a single payment "
            "exceeds Rs 30,000 or aggregate payments in a financial year exceed Rs 1,00,000. "
            "Section 194C also covers advertising contracts, catering, broadcasting, and "
            "carriage of goods. Sub-contractors of the primary contractor are also covered."
        ),
        "metadata": {"source": "Income Tax Act 1961", "section": "194C", "type": "legislation", "topic": "tds", "year": "1961"}
    },
    {
        "id": "tds_194j_001",
        "text": (
            "Section 194J covers TDS on fees for professional or technical services. TDS rate "
            "is 10% for professional services (lawyers, doctors, engineers, chartered accountants) "
            "and 2% for technical services. The threshold for deduction is Rs 30,000 per annum "
            "per payee. Professional services include medical, legal, engineering, and management "
            "consultancy. Technical services include managerial and consultancy services not "
            "covered under Section 194C."
        ),
        "metadata": {"source": "Income Tax Act 1961", "section": "194J", "type": "legislation", "topic": "tds", "year": "1961"}
    },
    {
        "id": "tds_return_001",
        "text": (
            "TDS returns must be filed quarterly. Form 24Q covers TDS on salaries, Form 26Q "
            "covers TDS on non-salary payments to residents, and Form 27Q covers TDS on "
            "payments to non-residents. Due dates: Q1 (Apr-Jun): July 31; Q2 (Jul-Sep): "
            "October 31; Q3 (Oct-Dec): January 31; Q4 (Jan-Mar): May 31. TDS certificate in "
            "Form 16 (salary) and Form 16A (non-salary) must be issued within 15 days of the "
            "due date of filing the quarterly TDS return."
        ),
        "metadata": {"source": "Income Tax Rules 1962", "section": "Rule 31A", "type": "rule", "topic": "tds", "year": "1962"}
    },

    # ── Income Tax ───────────────────────────────────────────────────────────
    {
        "id": "corp_tax_rates_001",
        "text": (
            "Corporate income tax rates for Indian domestic companies: 25% for companies with "
            "total turnover not exceeding Rs 400 crore in the preceding year; 30% for all other "
            "domestic companies. Section 115BAB provides a reduced rate of 15% for new domestic "
            "manufacturing companies incorporated after October 1, 2019. The effective rate "
            "including surcharge and health and education cess (4%) is approximately 25.17% "
            "for companies under the 25% slab."
        ),
        "metadata": {"source": "Income Tax Act 1961", "section": "115BAB", "type": "legislation", "topic": "corporate_tax", "year": "2019"}
    },
    {
        "id": "transfer_pricing_001",
        "text": (
            "Transfer pricing provisions under Sections 92 to 92F of the Income Tax Act require "
            "that international transactions between associated enterprises be at arm's length "
            "price. Methods: comparable uncontrolled price (CUP), resale price, cost plus, "
            "transactional net margin method (TNMM), and profit split. Taxpayers with "
            "international transactions exceeding Rs 1 crore must obtain a Transfer Pricing "
            "Report in Form 3CEB from a CA. Due date: October 31 of the assessment year."
        ),
        "metadata": {"source": "Income Tax Act 1961", "section": "92-92F", "type": "legislation", "topic": "transfer_pricing", "year": "1961"}
    },
    {
        "id": "advance_tax_001",
        "text": (
            "Advance tax is payable in four instalments when estimated tax liability exceeds "
            "Rs 10,000 in a financial year. Instalments: 15% by June 15, 45% cumulative by "
            "September 15, 75% cumulative by December 15, 100% by March 15. Interest under "
            "Section 234B applies at 1% per month for default in payment of advance tax. "
            "Interest under Section 234C applies for deferment of individual instalments. "
            "Senior citizens with no business income are exempt from advance tax."
        ),
        "metadata": {"source": "Income Tax Act 1961", "section": "208-219", "type": "legislation", "topic": "advance_tax", "year": "1961"}
    },
    {
        "id": "section_80c_001",
        "text": (
            "Section 80C of the Income Tax Act allows deductions up to Rs 1,50,000 per year "
            "from gross total income. Eligible investments and payments: life insurance premiums, "
            "PPF contributions, ELSS mutual funds, NSC, 5-year tax-saving fixed deposits, "
            "tuition fees for up to two children, EPF contributions, and principal repayment of "
            "home loans. This deduction is available only under the old tax regime. Under the "
            "new tax regime (Section 115BAC), Section 80C deductions are not available."
        ),
        "metadata": {"source": "Income Tax Act 1961", "section": "80C", "type": "legislation", "topic": "deductions", "year": "1961"}
    },
    {
        "id": "new_tax_regime_001",
        "text": (
            "New tax regime under Section 115BAC (FY 2025-26 slabs): Nil for income up to "
            "Rs 3 lakh, 5% for Rs 3-7 lakh, 10% for Rs 7-10 lakh, 15% for Rs 10-12 lakh, "
            "20% for Rs 12-15 lakh, 30% above Rs 15 lakh. Standard deduction of Rs 75,000 "
            "available. Rebate under Section 87A: zero tax for total income up to Rs 7 lakh. "
            "No Chapter VI-A deductions (80C, 80D etc.) or HRA exemption available. "
            "The new regime is the default from AY 2024-25."
        ),
        "metadata": {"source": "Income Tax Act 1961", "section": "115BAC", "type": "legislation", "topic": "income_tax_slabs", "year": "2023"}
    },
    {
        "id": "ltcg_001",
        "text": (
            "Long-term capital gains (LTCG) on listed equity shares and equity-oriented mutual "
            "funds exceeding Rs 1 lakh in a financial year are taxed at 10% without indexation "
            "benefit under Section 112A. Holding period for equity: more than 12 months. For "
            "unlisted shares and immovable property: more than 24 months. For debt mutual funds: "
            "more than 36 months. Short-term capital gains on listed equity are taxed at 15% "
            "under Section 111A. The Rs 1 lakh exemption cannot be carried forward."
        ),
        "metadata": {"source": "Income Tax Act 1961", "section": "112A", "type": "legislation", "topic": "capital_gains", "year": "1961"}
    },
    {
        "id": "itr_filing_001",
        "text": (
            "Income tax return filing due dates: July 31 for individuals, HUFs, and non-audit "
            "cases; October 31 for taxpayers subject to tax audit under Section 44AB and all "
            "companies; November 30 for transfer pricing cases. Belated return can be filed up "
            "to December 31 of the assessment year with late fee under Section 234F (Rs 5,000 "
            "for income above Rs 5 lakh, Rs 1,000 for income below Rs 5 lakh). "
            "Revised return can also be filed up to December 31."
        ),
        "metadata": {"source": "Income Tax Act 1961", "section": "139", "type": "legislation", "topic": "filing", "year": "1961"}
    },
    {
        "id": "gst_works_contract_001",
        "text": (
            "Works contract under GST is treated as a supply of service. GST rate for works "
            "contracts: 18% for general works contracts; 12% for construction of affordable "
            "housing projects, government-awarded civil structures, and roads. Pure labour "
            "contracts attract 18% GST. Input tax credit on goods and services used in works "
            "contracts is allowed to the contractor against output liability, subject to "
            "Section 17(5)(c) restrictions on immovable property construction."
        ),
        "metadata": {"source": "Notification 11/2017-CT(Rate)", "section": "Entry 3", "type": "notification", "topic": "works_contract", "year": "2017"}
    },
]

print(f"✅ {len(TAX_KNOWLEDGE_CHUNKS)} knowledge chunks loaded")
topics = sorted({c['metadata']['topic'] for c in TAX_KNOWLEDGE_CHUNKS})
print(f"   Topics covered: {topics}")

---
## 🔢 Step 5: Embedding Helper

In [ ]:
def get_embedding(text: str) -> list:
    """Call Azure OpenAI embeddings API. Returns a list of floats."""
    response = client.embeddings.create(
        model = AZURE_EMBEDDING_DEPLOYMENT,
        input = text
    )
    return response.data[0].embedding


# Quick smoke test
test_vec = get_embedding("GST on export of services")
print(f"✅ Embeddings API working — vector dimension: {len(test_vec)}")

---
## 🏗️ Step 6: Embed All Chunks and Build the Store

In [ ]:
# Initialise the store
tax_collection = NumpyVectorStore(name="tax_knowledge")

print(f"Embedding and indexing {len(TAX_KNOWLEDGE_CHUNKS)} chunks...\n")

for i, chunk in enumerate(TAX_KNOWLEDGE_CHUNKS, 1):
    vec = get_embedding(chunk["text"])
    tax_collection.add(
        ids        = [chunk["id"]],
        documents  = [chunk["text"]],
        embeddings = [vec],
        metadatas  = [{k: str(v) for k, v in chunk["metadata"].items()}],
    )
    print(f"  [{i:02d}/{len(TAX_KNOWLEDGE_CHUNKS)}] {chunk['id']}", end="\r")

print(f"\n\n✅ Vector store ready")
print(f"   Documents indexed : {tax_collection.count()}")
print(f"   Backend           : NumpyVectorStore (cosine similarity)")

---
## 🔍 Step 7: Retrieval Helper + 5 Semantic Queries

In [ ]:
def retrieve(query: str, n_results: int = 3, filter_metadata: dict = None) -> list:
    """
    Semantic retrieval. Embeds the query then returns the top-k
    most similar chunks with their similarity scores.
    """
    q_vec = get_embedding(query)
    results = tax_collection.query(
        query_embeddings = [q_vec],
        n_results        = n_results,
        where            = filter_metadata,
    )
    chunks = []
    for i in range(len(results["ids"][0])):
        chunks.append({
            "id":         results["ids"][0][i],
            "text":       results["documents"][0][i],
            "metadata":   results["metadatas"][0][i],
            "similarity": round(1 - results["distances"][0][i], 3),
        })
    return chunks


def print_retrieval(query: str, chunks: list):
    print(f"\n{'='*70}")
    print(f"  QUERY: {query}")
    print(f"{'='*70}")
    for i, c in enumerate(chunks, 1):
        m   = c["metadata"]
        bar = "█" * int(c["similarity"] * 20)
        print(f"\n  Rank {i}  [sim={c['similarity']:.3f}]  {bar}")
        print(f"  Source : {m.get('source','?')}  §{m.get('section','?')}")
        print(f"  Topic  : {m.get('topic','?')}  |  Type: {m.get('type','?')}")
        print(f"  Text   : {c['text'][:200]}...")


TEST_QUERIES = [
    "What is the GST rate on cloud software exported to a US company with LUT filed?",
    "When must I deduct TDS on a payment to a law firm, and at what rate?",
    "What are the income tax slabs for FY 2025-26 under the new tax regime?",
    "When is the GST return filing deadline for a monthly taxpayer?",
    "Can a company claim ITC on food expenses incurred for employees?",
]

for q in TEST_QUERIES:
    chunks = retrieve(q, n_results=3)
    print_retrieval(q, chunks)

---
## 🆚 Step 8: Semantic Retrieval vs Keyword Search

In [ ]:
def keyword_search(query: str, top_n: int = 3) -> list:
    """Naive keyword overlap search — no embeddings, no semantics."""
    query_words = set(re.findall(r'\b\w{4,}\b', query.lower()))
    scored = []
    for chunk in TAX_KNOWLEDGE_CHUNKS:
        text_words = set(re.findall(r'\b\w{4,}\b', chunk["text"].lower()))
        overlap    = len(query_words & text_words)
        score      = overlap / max(len(query_words), 1)
        scored.append((score, chunk["id"], chunk["text"][:120]))
    scored.sort(reverse=True)
    return scored[:top_n]


# Deliberately tricky query — uses different words from the documents
TRICKY = "Is the GST levy on digital consulting services to foreign clients zero-rated?"

print("SEMANTIC vs KEYWORD: side-by-side")
print(f"Query: {TRICKY}\n")

print("Semantic retrieval (NumpyVectorStore):")
for i, c in enumerate(retrieve(TRICKY, n_results=3), 1):
    print(f"  {i}. [sim={c['similarity']:.3f}]  {c['id']:<35}  {c['text'][:80]}...")

print()
print("Keyword search:")
for i, (score, chunk_id, snippet) in enumerate(keyword_search(TRICKY, top_n=3), 1):
    print(f"  {i}. [overlap={score:.3f}]  {chunk_id:<35}  {snippet}...")

print()
print("Observation: semantic search finds 'export of services' and 'zero-rated supply'")
print("even though the query says 'digital consulting' and 'foreign clients'.")
print("Keyword search may miss these because the exact words don't appear.")

---
## 🔎 Step 9: Inspect a Full Chunk

In [ ]:
# Inspect the top result for the export query in full
query   = "What GST rules apply to export of IT services with LUT?"
results = retrieve(query, n_results=1)
top     = results[0]

print("TOP RETRIEVED CHUNK — FULL CONTENT")
print("-" * 60)
print(f"Chunk ID   : {top['id']}")
print(f"Similarity : {top['similarity']:.3f}")
print(f"Source     : {top['metadata']['source']}")
print(f"Section    : {top['metadata']['section']}")
print(f"Topic      : {top['metadata']['topic']}")
print()
print("Full text:")
print(top['text'])
print()
print("This is exactly what gets injected into the prompt in Notebook 5C.")
print("The LLM generates its answer from THIS content — not from training memory.")

---
## 🎓 Key Insight

```
NumpyVectorStore stores:  embedding vector + document text + metadata
Query time:               embed query → cosine similarity over all vectors → return top-k
Result:                   3 highly relevant chunks ready to inject into a prompt
```

For 20 chunks this is instantaneous. For 20,000 chunks it's still fast (~100ms).
ChromaDB's HNSW index only outperforms linear scan above ~50,000 vectors.

---
## ➡️ Next: Notebook 5C — RAG Query Pipeline

We wire retrieval into generation:  
`question → embed → retrieve → augment prompt → generate cited answer`